[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/vectors/v03_space.ipynb)

# How does your phone know where it is? Points, distances and directions in 3D

A phone hears only radio messages from satellites, and each message gives it one distance. One distance is not a
place: it is a whole sphere of places. Three numbers say where a point is, Pythagoras done twice says how far apart
two points are, and the direction cosines say which way an arrow points.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the
▶ Run button) to run it. Then run each cell in turn, from top to bottom. When sliders appear, drag them. Change any
number you like and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in these notebooks. Run this cell first.
import math, warnings
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#8fcfc4", "#a9dca6", "#c3a9e6"   # the "along" parts · the result · the "across" parts

warnings.filterwarnings("ignore", message=".*as float was deprecated.*")   # a browser-only notice, harmless
plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def length(v):
    """Pythagoras done twice: across the floor, then up."""
    return math.sqrt(sum(x * x for x in v))

def space(ax, lim, view, elev=22):
    """A 3D axes on the slate: no grey panes, ghost grid, the camera at `view` degrees round the vertical."""
    ax.set_facecolor(SLATE)
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.set_pane_color((0, 0, 0, 0)); a._axinfo["grid"]["color"] = GHOST
    (x0, x1), (y0, y1), (z0, z1) = lim
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_zlim(z0, z1)
    ax.set_box_aspect((x1 - x0, y1 - y0, z1 - z0))
    ax.view_init(elev=elev, azim=view)
    ax.set_xlabel(""); ax.set_ylabel(""); ax.set_zlabel("")

def column(ax, vals, x, y, size=26, col=CHALK, name=""):
    """The arrow's numbers as a column in square brackets, drawn on a flat panel at (x, y) (top of the column)."""
    rows = [f"{v:.2f}".rstrip("0").rstrip(".").replace("-", "−") for v in vals]
    gap = 0.13
    for k, s in enumerate(rows):
        ax.text(x, y - k * gap, s, color=col, fontsize=size, ha="center", va="center", family="serif")
    top, bot, w = y + gap * 0.55, y - (len(rows) - 1) * gap - gap * 0.55, 0.13
    for side in (-1, 1):
        bx = x + side * w
        ax.plot([bx, bx], [bot, top], color=DIM, lw=2)
        ax.plot([bx, bx - side * 0.03], [top, top], color=DIM, lw=2)
        ax.plot([bx, bx - side * 0.03], [bot, bot], color=DIM, lw=2)
    if name: ax.text(x - w - 0.05, y - gap * (len(rows) - 1) / 2, name, color=col, fontsize=size, ha="right", va="center",
                     family="serif")

**The drawing tools.** Run the next cell too: it holds the code that draws each figure below. You don't need to read it
to play along.

In [ ]:
def room(x=2.0, y=1.0, z=2.0, view=-55):
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.0, 0.0, 0.62, 1.0], projection="3d")
    space(ax, ((0, 4), (0, 4), (0, 3)), view)
    for k in range(5):                                         # the floor grid, one metre squares
        ax.plot([k, k], [0, 4], [0, 0], color=GHOST, lw=1); ax.plot([0, 4], [k, k], [0, 0], color=GHOST, lw=1)
    for k in range(4):                                         # the two walls, faint
        ax.plot([0, 4], [0, 0], [k, k], color=GHOST, lw=0.8); ax.plot([0, 0], [0, 4], [k, k], color=GHOST, lw=0.8)
    for (dx, dy, dz), lab in (((4.3, 0, 0), "x"), ((0, 4.3, 0), "y"), ((0, 0, 3.3), "z")):
        ax.quiver(0, 0, 0, dx, dy, dz, color=DIM, lw=1.5, arrow_length_ratio=0.06)
        ax.text(dx * 1.04, dy * 1.04, dz * 1.04, lab, color=CHALK, fontsize=16)
    ax.plot([x, x], [y, y], [z, 0], ":", color=TEAL, lw=2)     # straight down to the floor
    ax.plot([x, x], [0, y], [0, 0], ":", color=TEAL, lw=2)     # and across to the edges
    ax.plot([0, x], [y, y], [0, 0], ":", color=TEAL, lw=2)
    ax.plot([0, x], [0, y], [0, 0], color=SAGE, lw=2.4)        # the floor diagonal
    ax.quiver(0, 0, 0, x, y, z, color=CHALK, lw=3, arrow_length_ratio=0.08)
    ax.scatter([x], [y], [z], color=CORAL, s=90, depthshade=False)
    floor, d = math.hypot(x, y), length((x, y, z))
    p = fig.add_axes([0.62, 0.0, 0.38, 1.0]); p.axis("off"); p.set_xlim(0, 1); p.set_ylim(0, 1)
    column(p, (x, y, z), 0.55, 0.78, name="r =")
    p.text(0.5, 0.36, f"floor: √({x:.1f}² + {y:.1f}²) = {floor:.2f}", color=SAGE, fontsize=14, ha="center")
    p.text(0.5, 0.26, f"then up: √({floor:.2f}² + {z:.1f}²)", color=CHALK, fontsize=14, ha="center")
    p.text(0.5, 0.12, f"distance from the corner = {d:.2f} m", color=CORAL, fontsize=17, ha="center")
    plt.show()

SATS = np.array([[-4.0, 1.5, 6.0], [3.5, -3.0, 6.5], [2.5, 4.0, 5.5], [-2.0, -4.5, 7.0]])   # far above the street
PHONE = np.array([0.0, 0.0, 0.0])

def sphere(ax, c, r, col, alpha):
    u, v = np.meshgrid(np.linspace(0, 2 * np.pi, 36), np.linspace(0, np.pi, 18))
    ax.plot_wireframe(c[0] + r * np.cos(u) * np.sin(v), c[1] + r * np.sin(u) * np.sin(v), c[2] + r * np.cos(v),
                      color=col, lw=0.6, alpha=alpha)

def circle_of(c1, r1, c2, r2):
    """Where two spheres meet: a circle (its centre, radius and two directions in its plane)."""
    n = c2 - c1; D = np.linalg.norm(n); n = n / D
    a = (r1 ** 2 - r2 ** 2 + D ** 2) / (2 * D)
    centre, rad = c1 + a * n, math.sqrt(max(r1 ** 2 - a ** 2, 0.0))
    u = np.cross(n, [0, 0, 1.0]); u /= np.linalg.norm(u); w = np.cross(n, u)
    return centre, rad, u, w

def satellites(n=1, view=-50):
    r = [np.linalg.norm(s - PHONE) for s in SATS]
    fig = plt.figure(figsize=(8, 5.4))
    ax = fig.add_axes([0.0, 0.0, 1.0, 0.93], projection="3d")
    space(ax, ((-11, 11), (-11, 11), (-4, 14)), view, elev=16)
    for k in range(-10, 11, 2):                                # the ground, as a faint grid
        ax.plot([k, k], [-10, 10], [0, 0], color=GHOST, lw=0.7); ax.plot([-10, 10], [k, k], [0, 0], color=GHOST, lw=0.7)
    for k in range(n):
        sphere(ax, SATS[k], r[k], CHALK if k < n - 1 else CORAL, 0.18 if k < n - 1 else 0.32)
        ax.scatter(*SATS[k], color=CHALK, s=70, marker="s", depthshade=False)
        ax.text(*(SATS[k] + [0, 0, 0.8]), f"satellite {k + 1}", color=CHALK, fontsize=11)
    title = "one distance: a whole sphere of places"
    if n == 2:
        c, rad, u, w = circle_of(SATS[0], r[0], SATS[1], r[1])
        t = np.linspace(0, 2 * np.pi, 200)
        ring = c[:, None] + rad * (np.outer(u, np.cos(t)) + np.outer(w, np.sin(t)))
        ax.plot(*ring, color=SAGE, lw=3)
        title = "two spheres meet in a circle"
    if n >= 3:
        nrm = np.cross(SATS[1] - SATS[0], SATS[2] - SATS[0]); nrm /= np.linalg.norm(nrm)
        mirror = PHONE - 2 * np.dot(PHONE - SATS[0], nrm) * nrm          # the other point: the phone's mirror image
        pts = [PHONE] if n == 4 else [PHONE, mirror]
        for q in pts: ax.scatter(*q, color=SAGE if n == 3 else CORAL, s=140, depthshade=False)
        if n == 3: ax.text(*(mirror + [0.5, 0, 0.5]), "far from the ground", color=DIM, fontsize=11)
        title = "three spheres: two points" if n == 3 else "four satellites (and the clock): one point"
    ax.scatter(*PHONE, color=CORAL, s=60, marker="^", depthshade=False)
    ax.text(*(PHONE + [0.6, 0, -1.6]), "phone", color=CORAL, fontsize=12)
    ax.set_title(f"{n} satellite{'s' if n > 1 else ''}: {title}", color=CORAL if n == 4 else CHALK)
    plt.show()

def turn(turn=18, tilt=25, size=7.0):
    a, e = math.radians(turn), math.radians(tilt)
    v = size * np.array([math.cos(e) * math.cos(a), math.cos(e) * math.sin(a), math.sin(e)])
    L = v / size
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.0, 0.0, 0.58, 1.0], projection="3d")
    space(ax, ((-7, 7), (-7, 7), (-7, 7)), -55, elev=20)
    for (dx, dy, dz), lab in (((7.5, 0, 0), "x"), ((0, 7.5, 0), "y"), ((0, 0, 7.5), "z")):
        ax.plot([-dx, dx], [-dy, dy], [-dz, dz], color=DIM, lw=1)
        ax.text(dx, dy, dz, lab, color=CHALK, fontsize=15)
    ax.plot([0, v[0]], [0, 0], [0, 0], color=TEAL, lw=4)      # the arrow's projection on each edge
    ax.plot([0, 0], [0, v[1]], [0, 0], color=TEAL, lw=4)
    ax.plot([0, 0], [0, 0], [0, v[2]], color=TEAL, lw=4)
    ax.quiver(0, 0, 0, *v, color=CHALK, lw=3, arrow_length_ratio=0.08)
    ax.quiver(0, 0, 0, *L, color=CORAL, lw=4, arrow_length_ratio=0.25)   # the unit vector: same way, length one
    p = fig.add_axes([0.58, 0.0, 0.42, 1.0]); p.axis("off"); p.set_xlim(0, 1); p.set_ylim(0, 1)
    column(p, v, 0.32, 0.82, size=22, name="a =")
    column(p, L, 0.78, 0.82, size=22, col=CORAL)
    p.text(0.32, 0.43, f"|a| = {size:.1f}", color=CHALK, fontsize=14, ha="center")
    p.text(0.78, 0.43, "÷ |a|  (length one)", color=CORAL, fontsize=14, ha="center")
    ang = [math.degrees(math.acos(max(-1, min(1, k)))) for k in L]
    p.text(0.5, 0.34, f"α = {ang[0]:.0f}°   β = {ang[1]:.0f}°   γ = {ang[2]:.0f}°", color=DIM, fontsize=14, ha="center")
    p.text(0.5, 0.24, f"l = {L[0]:.3f}   m = {L[1]:.3f}   n = {L[2]:.3f}".replace("-", "−"), color=TEAL, fontsize=15, ha="center")
    p.text(0.5, 0.11, f"l² + m² + n² = {np.sum(L ** 2):.3f}", color=CORAL, fontsize=19, ha="center")
    plt.show()

## 1 · A point in a room

Stand in the corner of a room, where two walls meet the floor. Three edges leave the corner: **x** along one wall,
**y** along the other, **z** straight up. A paper lantern hangs 2 m along, 1 m across and 2 m up, so its three numbers
are **(2, 1, 2)**, and no other point in the room has them.

Drag the sliders to move the point. The arrow from the corner and its column of numbers always say the same thing.
The teal dotted lines drop the point onto the floor and the edges; the sage line is the floor diagonal.

In [ ]:
interact(room, x=FloatSlider(value=2.0, min=0, max=4, step=0.5, description="x (m)"),
         y=FloatSlider(value=1.0, min=0, max=4, step=0.5, description="y (m)"),
         z=FloatSlider(value=2.0, min=0, max=3, step=0.5, description="z (m)"),
         view=IntSlider(value=-55, min=-90, max=0, step=5, description="view (°)"));

**Notice.** At (2, 1, 2) the floor diagonal is √5 ≈ 2.24 m, and then √(5 + 4) = √9 = 3 m. Walking along the edges
would be 2 + 1 + 2 = 5 m: the straight line is always shorter.

## 2 · From one point to another: head minus tail

When neither point is at the corner, subtract first: the arrow from P to S has the numbers S − P. Then use
Pythagoras twice on the differences. Change the two points and run the cell again.

In [ ]:
P = (1, 2, 0)          # the tail
S = (7, 4, 3)          # the head
d = tuple(s - p for s, p in zip(S, P))
print("S − P =", d)
print("distance =", f"√({' + '.join(f'{k}²' for k in d)}) = √{sum(k * k for k in d)} = {length(d):.4g}")
print("the drone: 6 m east, 2 m north, 3 m up →", length((6, 2, 3)), "m from take-off (its moves add to", 6 + 2 + 3, ")")

## 3 · Four satellites close in on one phone

Each satellite gives one distance, so it puts the phone somewhere on a sphere. Drag the slider to add satellites:
one sphere; two spheres meet in a sage circle; a third cuts the circle in two points (one far from the ground); a
fourth, which also settles the phone's clock, leaves one point. (Not to scale, and simplified: no relativity, no
least squares.)

In [ ]:
interact(satellites, n=IntSlider(value=1, min=1, max=4, step=1, description="satellites"),
         view=IntSlider(value=-50, min=-120, max=20, step=5, description="view (°)"));

**Notice.** The mirror point is found from the three satellites alone: it is the phone reflected in the flat sheet
through them, so it sits high above the ground. The fourth satellite is needed for the clock: four unknowns (x, y, z
and the clock error) need four distances.

## 4 · Which way? Direction cosines

An arrow makes three angles, α, β and γ, with the x, y and z edges. Their cosines **l, m, n** are the arrow's
numbers divided by its length: the share of the trip that goes along each edge. Turn the arrow any way you like with
the sliders, and change its length. Watch l, m and n change, while l² + m² + n² stays at one.

In [ ]:
interact(turn, turn=IntSlider(value=18, min=-180, max=180, step=6, description="turn (°)"),
         tilt=IntSlider(value=25, min=-85, max=85, step=5, description="tilt (°)"),
         size=FloatSlider(value=7.0, min=1, max=10, step=0.5, description="length"));

**Notice.** The length changes the arrow's numbers but not l, m and n: they depend only on the direction. That is
why direction ratios like (6, 2, 3), (12, 4, 6) and (60, 20, 30) all give the same cosines, 6/7, 2/7 and 3/7.

**The squares check.** Before you call three numbers direction cosines, square them and add: you must get one.

In [ ]:
def are_cosines(l, m, n):
    s = l * l + m * m + n * n
    print(f"({l:.4g})² + ({m:.4g})² + ({n:.4g})² = {s:.4g}".replace("-", "−"), "→ direction cosines" if abs(s - 1) < 1e-9 else "→ not cosines: divide by the length")

are_cosines(-2, -1, -2)
are_cosines(-2 / 3, -1 / 3, -2 / 3)

## Sources
GPS.gov, "Space Segment" and "GPS Overview" (satellites about 20,200 km up send their position and the time); NPTEL
"Basic GPS", lecture 8 (four unknowns: x, y, z and the clock). Alexis Clairaut, *Recherches sur les courbes à double
courbure* (Paris, 1731). NCERT Class XI Mathematics ch. 11 and Class XII ch. 10–11; CBSE Mathematics syllabus
2026–27. Retrieved 2026-10-26. The satellite picture is not to scale.